In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
from torch.utils.data import Dataset, DataLoader

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [7]:
# load test,train data
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [14]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [29]:
#processdata
drop = ["PassengerId","Name"]
catagorize = ["Sex","Embarked","Cabin"]

def process_and_filter(df,category = None):
    category_set = []
    df = df.drop(columns=[c for c in drop if c in df.columns]).copy()
    # for i in catagorize:
        # df[i] = df[i].astype("category").cat.codes +1
    # df[catagorize] = df[catagorize].astype("category").cat.codes+1
    if category:
        test[col] = pd.Categorical(test[col], categories=categories).codes + 1
        df[cols] = df[cols].apply(lambda col: col.astype("category").cat.codes + 1)
    else:
        df[cols] = df[cols].apply(lambda col: col.astype("category").cat.codes + 1)
    df = df.fillna(df.mean(numeric_only=True))

    return df , category_set

train , category_set = process_and_filter(train)
# categories = train.cat.categories
test = process_and_filter(test)
# test[col] = pd.Categorical(test[col], categories=categories).codes + 1
train.head()

AttributeError: 'DataFrame' object has no attribute 'cat'

In [27]:
test.head()

,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,2,34.5,0,0,330911,7.8292,1,2
1,3,1,47.0,1,0,363272,7.0000,1,3
2,2,2,62.0,0,0,240276,9.6875,1,2
3,3,2,27.0,0,0,315154,8.6625,1,3
4,3,1,22.0,1,1,3101298,12.2875,1,3


In [13]:
class DataFrameDataset(Dataset):
    def __init__(self,df,target:str ):
        self.features = torch.tensor(df.drop(columns=[target]).values,dtype = torch.float32)
        self.labels = torch.tensor(df[target].values,dtype=torch.float32)
    def __len__(self):
        return len(features)
        
    def __getitem(self,idx):
        return self.features,self.labels

train_torch_dataset = DataFrameDataset(train,"Survived")
dataloader = DataLoader(train_torch_dataset,batch_size=32,shuffle=True)


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.